In [1]:
import numpy as np
import gymnasium as gym
from gymnasium.utils.env_checker import check_env

In [2]:
def simulate_tracks(
    n_planes,
    spatial_resolution,
    n_tracks=1000,
    plane_size=100.0,
    start_area_fraction=0.10,
    z_min=0.0,
    z_max=600.0,
    theta_max_deg=10.0,
    beta_min=0.5,
    beta_max=0.75,
    time_resolution=2.0,
    tof_margin=50.0,
    default_value=-999999.0,
    seed=42
):
    """
    Simula tracce Monte Carlo con:
    - n_planes piani di tracciamento sensibili a x,y
    - 2 piani temporali, uno prima e uno dopo il tracciatore
    - velocità casuale tra beta_min*c e beta_max*c
    - controllo dell'accettanza geometrica dei piani

    Le hit fuori accettanza vengono riempite con default_value.

    Returns
    -------
    x_meas : np.ndarray
        Coordinate x misurate, shape = (n_tracks, n_planes).
        Se fuori accettanza: default_value.

    y_meas : np.ndarray
        Coordinate y misurate, shape = (n_tracks, n_planes).
        Se fuori accettanza: default_value.

    z_planes : np.ndarray
        Coordinate z dei piani di tracciamento, shape = (n_planes,)

    theta : np.ndarray
        Angolo polare generato rispetto all'asse z, in radianti.

    theta_deg : np.ndarray
        Angolo polare generato rispetto all'asse z, in gradi.

    beta : np.ndarray
        Velocità generata in unità di c.

    delta_t_meas : np.ndarray
        Differenza di tempo misurata tra i due piani temporali, in ns.

    hit_valid : np.ndarray
        Maschera booleana, shape = (n_tracks, n_planes).
        True se la hit è dentro l'accettanza del piano.
    """

    rng = np.random.default_rng(seed)

    # Velocità della luce in mm/ns
    c_mm_ns = 299.792458

    # Posizione dei piani di tracciamento
    z_planes = np.linspace(z_min, z_max, n_planes)

    # Posizione dei due piani temporali
    z_time_1 = z_min - tof_margin
    z_time_2 = z_max + tof_margin

    delta_z_tof = z_time_2 - z_time_1

    # Regione iniziale pari a start_area_fraction dell'area del piano
    start_size = plane_size * np.sqrt(start_area_fraction)
    start_half_size = start_size / 2

    x0 = rng.uniform(-start_half_size, start_half_size, n_tracks)
    y0 = rng.uniform(-start_half_size, start_half_size, n_tracks)

    # Angoli casuali entro un cono attorno all'asse z
    theta_max = np.deg2rad(theta_max_deg)

    phi = rng.uniform(0, 2 * np.pi, n_tracks)

    # Distribuzione uniforme in angolo solido
    cos_theta = rng.uniform(np.cos(theta_max), 1, n_tracks)
    theta = np.arccos(cos_theta)
    theta_deg = np.rad2deg(theta)

    tan_theta = np.tan(theta)

    tx = tan_theta * np.cos(phi)   # dx/dz
    ty = tan_theta * np.sin(phi)   # dy/dz

    # Velocità casuale
    beta = rng.uniform(beta_min, beta_max, n_tracks)

    # Hit vere sui piani di tracciamento
    x_true = x0[:, None] + tx[:, None] * z_planes[None, :]
    y_true = y0[:, None] + ty[:, None] * z_planes[None, :]

    # Maschera di accettanza geometrica
    half_plane_size = plane_size / 2

    hit_valid = (
        (np.abs(x_true) <= half_plane_size) &
        (np.abs(y_true) <= half_plane_size)
    )

    # Hit misurate con risoluzione spaziale gaussiana
    x_meas = x_true + rng.normal(
        0,
        spatial_resolution,
        size=x_true.shape
    )

    y_meas = y_true + rng.normal(
        0,
        spatial_resolution,
        size=y_true.shape
    )

    # Se la traccia è fuori accettanza, il piano non misura nessuna hit
    x_meas[~hit_valid] = default_value
    y_meas[~hit_valid] = default_value

    # Tempo di volo vero tra i due piani temporali
    path_length = delta_z_tof / np.cos(theta)
    delta_t_true = path_length / (beta * c_mm_ns)

    # Misura del tempo sui due piani temporali
    t1_meas = rng.normal(0, time_resolution, n_tracks)
    t2_meas = delta_t_true + rng.normal(0, time_resolution, n_tracks)

    delta_t_meas = t2_meas - t1_meas

    return x_meas, y_meas, z_planes, theta, theta_deg, beta, delta_t_meas

In [ ]:
import numpy as np
from gymnasium import spaces

class DetectorOptimizerEnv(gym.Env):
    def __init__(self):
        super().__init__()

        # --- SPAZIO DELLE AZIONI ---
        # [n_piani, risoluzione_spaziale, risoluzione_temporale, lunghezza_totale_z]
        # n_piani: da 3 a 20
        # risoluzione_spaziale: da 0.01 a 1.0 mm
        # risoluzione_temporale: da 0.1 a 5.0 ns
        # lunghezza_totale (z_max): da 100 a 1000 mm
        self.action_space = spaces.Box(
            low=np.array([2.0, 0.001, 0.01, 10.0]),
            high=np.array([20.0, 1.0, 5.0, 1000.0]),
            dtype=np.float32
        )

        # L'osservazione è lo stato del design attuale
        self.observation_space = self.action_space

        # Stato iniziale casuale
        self.state = self.action_space.sample()

    def step(self, action):
        # 1. Parsing delle azioni
        n_planes = int(np.clip(np.round(action[0]), 2, 20))
        spatial_res = float(np.clip(action[1], 0.01, 1))
        time_res = float(np.clip(action[2], 0.1, 5))
        z_max_val = float(np.clip(action[3], 100, 1000))

        # 2. Esecuzione Simulazione Monte Carlo
        # (Usiamo la tua funzione simulate_tracks adattata)
        results = self._run_simulation(n_planes, spatial_res, time_res, z_max_val)
        
        # 3. Calcolo del Reward
        # Vogliamo minimizzare l'errore su beta (massimizzare -MSE)
        mse_beta = np.mean((results['beta_meas'] - results['beta_true'])**2)
        
        # Introduciamo un "Penalty Cost" per evitare design infinitamente costosi
        # Più piani = più costo | Risoluzione piccola = più costo
        #cost = (n_planes * 1) + (spatial_res ** 1.5) + (time_res * 1)
        
       # 3. Nuovo Calcolo del Reward (Più stabile)
        # Penalizziamo la precisione (valori piccoli = costo alto)
        cost_n = n_planes * 0.5
        cost_res = (1.0 / spatial_res) * 0.1  # 0.01 mm costa 10, 1.0 mm costa 0.1
        cost_time = (1.0 / time_res) * 0.5   # 0.1 ns costa 5, 5 ns costa 0.1
        total_cost =  mse_beta + cost_n + cost_res + cost_time
        
        # Usiamo una funzione più dolce del log10 per la precisione
        # Clipping del reward per evitare esplosioni
        precision_reward = -np.log10(mse_beta + 1e-4) 
        reward = precision_reward - total_cost
        reward = np.clip(reward, -100, 100) # CINTURA DI SICUREZZA FINALE

        self.state = action
        return self.state, float(reward), False, False, {"mse_beta": mse_beta, "n_planes": n_planes}

    def _run_simulation(self, n_p, s_res, t_res, z_max):
        # Chiamata alla funzione originale fornita da te
        # Assumiamo n_tracks=200 per avere una statistica decente nel reward
        x_m, y_m, z_p, th, th_d, beta_true, dt_m = simulate_tracks(
            n_planes=n_p,
            spatial_resolution=s_res,
            time_resolution=t_res,
            z_max=z_max,
            n_tracks=200
        )
        
        # Ricostruzione brutale di beta per valutare il design
        c_mm_ns = 299.792458
        z_time_1 = 0.0 - 50.0 # tof_margin fisso
        z_time_2 = z_max + 50.0
        path_length = (z_time_2 - z_time_1) / np.cos(th)
        beta_meas = path_length / (dt_m * c_mm_ns)
        
        return {'beta_true': beta_true, 'beta_meas': beta_meas}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.action_space.sample()
        return self.state, {}

In [4]:
import torch
from torchrl.envs.libs.gym import GymWrapper
from torchrl.envs import TransformedEnv, StepCounter

# 1. Creazione dell'ambiente
base_env = DetectorOptimizerEnv()
env = GymWrapper(base_env)

batch_size = 256
ep_length = 4096
# 2. Aggiunta di trasformazioni
# Poiché l'azione ha range diversi (3-20 vs 0.01-1), 
# è utile normalizzarla tra -1 e 1 per la rete neurale.
#from torchrl.envs.transforms import DoubleToFloat, ActionScaling

env = TransformedEnv(env)
env.append_transform(StepCounter(max_steps=ep_length))

# Esempio di interazione:
# Supponiamo che la policy generi un'azione
action = torch.tensor([0.5, 0.8, 0.2, 0.1]) # Azione normalizzata

/home/stelfano/anaconda3/envs/RL/lib/python3.13/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/stelfano/anaconda3/envs/RL/lib/python3.13/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [5]:
rollout = env.rollout(1)
print("action:", rollout['action'])
print("reward:", rollout['next', 'reward'])
print("observation:", rollout['next', 'observation'])
print("initial observation:", rollout['observation'])

action: tensor([[3.7471e+00, 5.1194e-01, 3.7068e+00, 8.4385e+02]])
reward: tensor([[-100.]])
observation: tensor([[3.7471e+00, 5.1194e-01, 3.7068e+00, 8.4385e+02]])
initial observation: tensor([[1.1895e+01, 3.2928e-01, 3.9951e+00, 4.0680e+02]])


In [7]:
from torchrl.trainers.algorithms import SACTrainer
from torchrl.modules import MLP, Actor, ProbabilisticActor, ValueOperator, SafeModule, IndependentNormal
from torchrl.modules.distributions import TanhNormal 
from torchrl.collectors import Collector, SyncDataCollector
from torchrl.data import Bounded, ReplayBuffer, LazyTensorStorage, TensorDictReplayBuffer
from torchrl.objectives import SACLoss, SoftUpdate
from tensordict.nn import TensorDictModule, NormalParamExtractor
from torchrl.envs.utils import ExplorationType
from torch import nn
import torch.optim as optim
from tensordict import TensorDict

n_actions = env.action_space.shape[0]
n_obs = env.observation_space.shape[0]

spec = Bounded(torch.tensor([2.0, 0.001, 0.01, 10.0]), torch.tensor([20.0, 1.0, 5.0, 1000.0]))

net = nn.Sequential(
    nn.Linear(n_obs, 128),
    nn.ReLU(inplace=False),
    nn.Linear(128, 256),
    nn.ReLU(inplace=False),
    nn.Linear(256, 1024),
    nn.ReLU(inplace=False),
    nn.Linear(1024, 128),
    nn.ReLU(inplace=False),
    nn.Linear(128, 2 * n_actions), # Output doppio per medie e deviazioni standard
    NormalParamExtractor(),    # Separa l'output in "loc" e "scale"
)

policyMLP = TensorDictModule(
    module=net,
    in_keys=["observation"],
    out_keys=["loc", "scale"]
)

actor = ProbabilisticActor(
    module=policyMLP,
    distribution_class=IndependentNormal,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    return_log_prob=True,
    cache_dist = False
)


critic_net = MLP(
    in_features=n_obs + n_actions,
    out_features=1,
    num_cells=[256, 1024, 2048, 1024, 256],
    activation_class=torch.nn.ReLU,
    activate_last_layer=False
)


# 2. Avvolgi la MLP in un ValueOperator
# Questo aggiunge gli attributi 'in_keys' e 'out_keys' richiesti dalla loss
critic_module = ValueOperator(
    module=critic_net,
    in_keys=["observation", "action"],
    out_keys=["state_action_value"] # Chiave standard per SAC
)

# 3. Ora passa critic_module (NON critic_net) alla loss
loss_module = SACLoss(
    actor_network=actor,
    qvalue_network=critic_module,
    action_spec=env.action_spec,
    num_qvalue_nets=1, # TorchRL sdoppierà critic_module internamente
    loss_function="smooth_l1",
)


In [8]:
frames_per_batch = 128
collector = Collector(
    env,               # Il tuo ambiente Gym/TorchRL
    policy=actor,                    # L'attore probabilistico definito prima
    frames_per_batch=frames_per_batch,           # Quanti frame raccogliere prima di ogni aggiornamento
    total_frames=ep_length,             # Durata totale dell'esperimento
)


In [9]:
from torchrl.record.loggers.csv import CSVLogger
from torchrl.trainers import UpdateWeights, LogScalar
optimizer = optim.Adam(loss_module.parameters(), lr=1e-5)
replay_buffer = ReplayBuffer(storage=LazyTensorStorage(100000), batch_size=batch_size)
net_updater = SoftUpdate(loss_module, eps=0.05)


/home/stelfano/anaconda3/envs/RL/lib/python3.13/site-packages/torchrl/objectives/utils.py:339: UserWarning: Found an eps value < 0.5, which is unexpected. You may want to use the `tau` keyword argument instead.
  warnings.warn(


In [ ]:
import torch
from torchrl.data import ReplayBuffer, ListStorage
from tqdm import tqdm

# --- CONFIGURAZIONE IPERPARAMETRI ---
max_grad_norm = 0.5      # Clipping aggressivo per stabilità
reward_scale = 0.2       # Porta i tuoi reward da ~ -10 a ~ -1.0
init_random_steps = 1000 # Raccogli più dati prima di addestrare
utd_ratio = 1            # Update-to-Data ratio (quanti update per ogni step)
device = "cpu"

# Inizializzazione Buffer
replay_buffer = ReplayBuffer(storage=ListStorage(max_size=100000))

print("Inizio raccolta dati iniziale...")

for i, data in enumerate(collector):
    
    # 1. PRE-PROCESSING DEI DATI
    # Applichiamo il reward scaling prima di salvare nel buffer
    data["next", "reward"] = data["next", "reward"] * reward_scale
    #Print the action and reward for debugging
    print(f"Step {i} | Action: {data['action']} | Reward: {data['next', 'reward']}") # Debugging: stampa azione e reward
    
    # 2. AGGIORNAMENTO BUFFER
    current_frames = data.numel()
    replay_buffer.extend(data)
    
    # Iniziamo l'ottimizzazione solo dopo i passi random iniziali
    if len(replay_buffer) > init_random_steps:
        
        # tqdm per monitorare i mini-batch
        for _ in range(utd_ratio):
            
            # 3. CAMPIONAMENTO
            sample = replay_buffer.sample(batch_size)
            if sample.device != device:
                sample = sample.to(device)

            # 4. CALCOLO LOSS
            loss_vals = loss_module(sample)
            actor_loss = loss_vals["loss_actor"]
            q_loss = loss_vals["loss_qvalue"]
            alpha_loss = loss_vals["loss_alpha"] # Se usi auto-entropy
            
            total_loss = actor_loss + q_loss + alpha_loss

            # 5. BACKWARD E CLIPPING (La parte cruciale)
            optimizer.zero_grad()
            total_loss.backward()
            
            # Applichiamo il clipping separatamente per sicurezza
            torch.nn.utils.clip_grad_norm_(loss_module.actor_network_params.parameters(), max_grad_norm)
            torch.nn.utils.clip_grad_norm_(loss_module.qvalue_network_params.parameters(), max_grad_norm)
            
            # 6. STEP DI OTTIMIZZAZIONE
            optimizer.step()

            # 7. SOFT UPDATE DEL TARGET (Target Networks)
            net_updater.step()

    # --- LOGGING ---
            avg_reward = data["next", "reward"].mean().item() / reward_scale # Torna alla scala reale per il log
            print(f"Batch {i} | Reward: {avg_reward:.2f} | Loss Actor: {actor_loss:.2f} | Loss Q: {q_loss:.2f}")

    # Condizione di stop (es. numero di episodi o performance)
    if i > 5000: 
        break